In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import os

from stable_baselines3 import PPO

from week4_trading_env_v2 import TradingEnvV2



def run_episode(
    env,
    model=None
):


    obs, _ = env.reset()

    rewards = []

    done = False


    while not done:


        if model is None:

            action = env.action_space.sample()

        else:

            action, _ = model.predict(
                obs,
                deterministic=True
            )


        obs, reward, terminated, truncated, _ = env.step(action)


        rewards.append(
            reward
        )


        done = terminated or truncated



    return rewards




def buy_and_hold_episode(env):


    obs, _ = env.reset()

    rewards = []

    done=False


    while not done:


        obs, reward, terminated, truncated, _ = env.step(0)


        rewards.append(
            reward
        )


        done = terminated or truncated



    return rewards




def average_cumulative_pnl(
    env,
    n_episodes=50,
    model=None,
    strategy="random"
):


    all_curves=[]


    for i in range(n_episodes):


        if strategy=="buy_and_hold":

            rewards = buy_and_hold_episode(env)

        else:

            rewards = run_episode(
                env,
                model
            )


        all_curves.append(
            np.cumsum(rewards)
        )


    length = min(
        len(x)
        for x in all_curves
    )


    values=np.array(
        [
            x[:length]
            for x in all_curves
        ]
    )


    return values.mean(axis=0)




def plot_pnl(
    trained,
    random,
    bnh
):


    os.makedirs(
        "plots",
        exist_ok=True
    )


    plt.figure(
        figsize=(9,4)
    )


    plt.plot(
        trained,
        label="PPO"
    )


    plt.plot(
        random,
        label="Random"
    )


    plt.plot(
        bnh,
        label="Buy and Hold"
    )


    plt.xlabel("Step")

    plt.ylabel("Cumulative P&L")


    plt.title(
        "TradingEnvV2 comparison"
    )


    plt.legend()


    plt.tight_layout()


    plt.savefig(
        "plots/cumulative_pnl.png"
    )


    plt.close()



if __name__=="__main__":


    env = TradingEnvV2()


    model = PPO(
        "MlpPolicy",
        env,
        verbose=0,
        learning_rate=3e-4,
        n_steps=512,
        batch_size=64,
        gamma=0.99
    )


    model.learn(
        total_timesteps=100000
    )



    test_env = TradingEnvV2()



    ppo_curve = average_cumulative_pnl(
        test_env,
        model=model,
        strategy="trained"
    )


    random_curve = average_cumulative_pnl(
        test_env
    )


    bnh_curve = average_cumulative_pnl(
        test_env,
        strategy="buy_and_hold"
    )


    print(
        "PPO:",
        ppo_curve[-1]
    )


    print(
        "Random:",
        random_curve[-1]
    )


    print(
        "Buy Hold:",
        bnh_curve[-1]
    )


    plot_pnl(
        ppo_curve,
        random_curve,
        bnh_curve
    )